# EmoFace: improve the AlexNet with real photos from Hugging Face (Google Colab)

This notebook takes the model you already have (`alexnet_emotion.pth`) and **fine-tunes it on real face photos downloaded from Hugging Face** (`Mauregato/affectnet_short`, about 29,000 labelled faces taken in everyday conditions). That is closer to a live webcam than the small 48x48 FER-2013 pictures, so smiles, sad faces and surprise are recognised more reliably and the model stops defaulting to *neutral*.

**Steps**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Run the cells from top to bottom (about 10 to 15 minutes).
3. When you are asked, choose your current `alexnet_emotion.pth` (the one in your project's `models` folder).
4. At the end the new `alexnet_emotion.pth` downloads. Replace the old file in `models/` and push to GitHub. Render redeploys by itself.

The cell near the end prints the accuracy of the **old and the new model on the same photos the training never saw**, so you can see the improvement as numbers.

> Dataset note: AffectNet-derived data is meant for research and education. Fine for a college mini-project; do not use it for a commercial product.

In [ ]:
!pip -q install datasets

In [ ]:
import os, io, copy, time, random
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision import models
from datasets import load_dataset

CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]   # same order the website uses
HF_DATASET = "Mauregato/affectnet_short"

MAX_TRAIN_PER_CLASS = 4000     # stops happy/neutral from dominating
MAX_VAL_PER_CLASS   = 500
EPOCHS      = 6
BATCH       = 128
LR          = 6e-5
DISTILL_W   = 0.5              # how strongly to stay close to your current model (keeps what already works)
SEED        = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

## 1. Your current model

In [ ]:
CKPT_IN = "alexnet_emotion.pth"
if not os.path.exists(CKPT_IN):
    from google.colab import files
    print("Choose your current alexnet_emotion.pth")
    files.upload()

def build_alexnet(n=7):
    m = models.alexnet(weights=None)
    m.classifier[6] = nn.Linear(4096, n)
    return m

ckpt = torch.load(CKPT_IN, map_location="cpu", weights_only=False)
state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
assert list(ckpt.get("classes", CLASSES)) == CLASSES, "class order differs from the website's"
teacher = build_alexnet(); teacher.load_state_dict(state); teacher.to(dev).eval()
for p in teacher.parameters(): p.requires_grad_(False)
model = copy.deepcopy(teacher)
for p in model.parameters(): p.requires_grad_(True)
print("loaded; old validation accuracy stored in file:", ckpt.get("fer2013_val_acc"))

## 2. Download real photos from Hugging Face

The dataset's own label names are used to map every photo to the website's seven emotions. The *contempt* label is dropped because the website does not have it.

In [ ]:
NAME_MAP = {"anger": "angry", "angry": "angry", "disgust": "disgust", "fear": "fear",
            "happy": "happy", "happiness": "happy", "neutral": "neutral",
            "sad": "sad", "sadness": "sad", "surprise": "surprise"}   # contempt -> dropped
FALLBACK_IDS = {0: "angry", 1: "surprise", 3: "happy", 4: "neutral", 5: "fear", 6: "sad", 7: "disgust"}  # Mauregato/affectnet_short

raw = load_dataset(HF_DATASET)
print(raw)
train_raw, val_raw = raw["train"], raw["val"] if "val" in raw else raw["validation"]
feat = train_raw.features["label"]
if hasattr(feat, "names"):
    ID2CLS = {i: NAME_MAP.get(n.lower()) for i, n in enumerate(feat.names)}
else:
    ID2CLS = FALLBACK_IDS
print("label mapping:", {i: c for i, c in ID2CLS.items()})
assert set(c for c in ID2CLS.values() if c) == set(CLASSES), "could not map all seven emotions"

In [ ]:
def to_gray48(img):
    g = cv2.cvtColor(np.asarray(img.convert("RGB")), cv2.COLOR_RGB2GRAY)
    return cv2.resize(g, (48, 48), interpolation=cv2.INTER_AREA)       # same reduction the website applies

def prepare(split, cap, seed):
    rng = np.random.RandomState(seed)
    labels = np.array(split["label"])
    keep = []
    for cid, cname in ID2CLS.items():
        if cname is None: continue
        idx = np.where(labels == cid)[0]
        rng.shuffle(idx); keep += list(idx[:cap])
    keep = sorted(keep)
    X = np.stack([to_gray48(split[int(i)]["image"]) for i in keep])
    y = np.array([CLASSES.index(ID2CLS[int(labels[i])]) for i in keep])
    return X, y

t0 = time.time()
Xtr, ytr = prepare(train_raw, MAX_TRAIN_PER_CLASS, SEED)
Xva, yva = prepare(val_raw, MAX_VAL_PER_CLASS, SEED + 1)
print(f"prepared in {time.time()-t0:.0f}s   train {Xtr.shape}   val {Xva.shape}")
print("train per class:", dict(zip(CLASSES, np.bincount(ytr, minlength=7))))
print("val   per class:", dict(zip(CLASSES, np.bincount(yva, minlength=7))))

## 3. Training pictures that look like a webcam

Every training photo is randomly shifted, rotated, zoomed, mirrored, darkened or brightened, softened and given a little noise. The website applies exactly the same contrast stretch before the model sees a face, so training uses it too.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def autocontrast_t(x, cutoff=0.01):
    # x: (B,1,48,48) in 0..1. Same idea as PIL ImageOps.autocontrast(cutoff=1).
    flat = x.flatten(1)
    s, _ = flat.sort(dim=1)
    n = flat.shape[1]; k = int(n * cutoff)
    lo, hi = s[:, k:k+1], s[:, n-1-k:n-k]
    scale = 255.0 / (hi - lo).clamp(min=1e-3) / 255.0
    return ((flat - lo) * scale).clamp(0, 1).view_as(x)

def to_input(x01):
    x = F.interpolate(x01, size=227, mode="bicubic", align_corners=False).clamp(0, 1)
    return ((x.repeat(1, 3, 1, 1) - MEAN) / STD)

def augment(xb):
    B = xb.shape[0]
    ang = (torch.rand(B, device=dev) - .5) * 2 * np.deg2rad(12)
    sc  = 1 + (torch.rand(B, device=dev) - .5) * 0.24
    tx, ty = [(torch.rand(B, device=dev) - .5) * 0.16 for _ in range(2)]
    flip = (torch.rand(B, device=dev) < .5).float() * 2 - 1
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos * flip, -sin, tx], 1), torch.stack([sin * flip, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, xb.shape, align_corners=False)
    x = F.grid_sample(xb, grid, padding_mode="border", align_corners=False)
    gamma = torch.exp((torch.rand(B, 1, 1, 1, device=dev) - .5) * 1.0)        # lighting
    x = x.clamp(1e-4, 1) ** gamma
    x = (x - .5) * (0.65 + torch.rand(B, 1, 1, 1, device=dev) * .7) + .5 + (torch.rand(B, 1, 1, 1, device=dev) - .5) * .25
    blur = torch.rand(B, device=dev) < .35                                       # soft focus
    if blur.any():
        k = torch.tensor([1., 2., 1.], device=dev); k = (k[:, None] * k[None, :]); k = (k / k.sum()).view(1, 1, 3, 3)
        x[blur] = F.conv2d(F.pad(x[blur], (1, 1, 1, 1), mode="replicate"), k)
    x = x + torch.randn_like(x) * (torch.rand(B, 1, 1, 1, device=dev) * .04)      # sensor noise
    return x.clamp(0, 1)

def eval_inputs(X48):
    # Exactly what the website does: PIL autocontrast, bicubic to 227, ImageNet normalisation.
    out = []
    for g in X48:
        img = ImageOps.autocontrast(Image.fromarray(g), cutoff=1).resize((227, 227), Image.BICUBIC)
        out.append(np.asarray(img))
    t = torch.from_numpy(np.stack(out)).float().div(255).unsqueeze(1)
    return ((t.repeat(1, 3, 1, 1) - MEAN.cpu()) / STD.cpu())

print("preparing validation tensors ...")
Xva_in = eval_inputs(Xva)
print(Xva_in.shape)

## 4. Fine-tune (the first three convolution layers stay frozen)

In [ ]:
@torch.no_grad()
def logits_of(net, X_in, bs=128):
    # Average of the photo and its mirror image, like the website.
    net.eval(); outs = []
    for i in range(0, len(X_in), bs):
        b = X_in[i:i+bs].to(dev)
        outs.append(((net(b) + net(torch.flip(b, dims=[3]))) / 2).float().cpu())
    return torch.cat(outs)

def report(lg, y, name):
    pred = lg.argmax(1).numpy()
    acc = (pred == y).mean()
    rec = [(pred[y == c] == c).mean() if (y == c).any() else float("nan") for c in range(7)]
    bal = np.nanmean(rec)
    print(f"{name:>10}: accuracy {acc*100:5.1f}%   balanced {bal*100:5.1f}%   per class:",
          " ".join(f"{CLASSES[c][:3]} {rec[c]*100:3.0f}" for c in range(7)))
    return acc, bal

old_val_logits = logits_of(teacher, Xva_in)
report(old_val_logits, yva, "OLD model")

for p in model.features[:8].parameters(): p.requires_grad_(False)       # conv1..conv3 frozen
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
steps = EPOCHS * int(np.ceil(len(Xtr) / BATCH))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.2)
cw = torch.tensor(len(ytr) / (7 * np.bincount(ytr, minlength=7)), dtype=torch.float32, device=dev)
ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
scaler = torch.amp.GradScaler(enabled=(dev.type == "cuda"))

Xtr_t = torch.from_numpy(Xtr).float().div(255).unsqueeze(1).to(dev)
ytr_t = torch.from_numpy(ytr).to(dev)
T = 2.0
best, best_state = -1, None
for ep in range(1, EPOCHS + 1):
    model.train(); perm = torch.randperm(len(Xtr_t), device=dev); tot = 0; t0 = time.time()
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i+BATCH]
        x = to_input(autocontrast_t(augment(Xtr_t[idx]))); y = ytr_t[idx]
        with torch.autocast(device_type=dev.type, enabled=(dev.type == "cuda")):
            out = model(x)
            with torch.no_grad(): tl = teacher(x)
            loss = ce(out.float(), y) + DISTILL_W * T * T * F.kl_div(
                F.log_softmax(out.float() / T, 1), F.softmax(tl.float() / T, 1), reduction="batchmean")
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(idx)
    lg = logits_of(model, Xva_in)
    acc, bal = report(lg, yva, f"epoch {ep}")
    print(f"           loss {tot/len(Xtr_t):.3f}   {time.time()-t0:.0f}s")
    if bal > best: best, best_state = bal, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print("kept the best epoch (balanced accuracy %.1f%%)" % (best * 100))

## 5. Balance the predictions and compare old vs new

Half of the held-out photos are used to nudge each emotion's score (so no emotion is over-predicted), and the other half, which this step never touched, is used for the honest comparison.

In [ ]:
rng = np.random.RandomState(1); order = rng.permutation(len(yva))
cal, tst = order[: len(order)//2], order[len(order)//2 :]
new_logits = logits_of(model, Xva_in)

def bal_acc(lg, y):
    p = lg.argmax(1).numpy()
    return np.nanmean([(p[y == c] == c).mean() for c in range(7) if (y == c).any()])

bias = np.zeros(7, np.float32)
for _ in range(3):
    for c in range(7):
        scores = []
        for d in np.arange(-0.6, 0.61, 0.05):
            b = bias.copy(); b[c] = d
            scores.append((bal_acc(new_logits[cal] + torch.from_numpy(b), yva[cal]) - 0.02 * abs(d), d))
        bias[c] = max(scores)[1]
bias -= bias[CLASSES.index("neutral")]            # neutral is the reference
print("bias:", dict(zip(CLASSES, np.round(bias, 2))))

print("\nOn photos neither model trained on or was calibrated with:")
report(old_val_logits[tst], yva[tst], "OLD model")
report(new_logits[tst] + torch.from_numpy(bias), yva[tst], "NEW model")

import matplotlib.pyplot as plt
from itertools import product
pred = (new_logits[tst] + torch.from_numpy(bias)).argmax(1).numpy()
cm = np.zeros((7, 7), int)
for t_, p_ in zip(yva[tst], pred): cm[t_, p_] += 1
plt.figure(figsize=(6, 5)); plt.imshow(cm / cm.sum(1, keepdims=True).clip(1), cmap="Blues")
plt.xticks(range(7), CLASSES, rotation=45); plt.yticks(range(7), CLASSES)
for i, j in product(range(7), range(7)): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("New model (held-out photos)"); plt.tight_layout(); plt.show()

## 6. Save the new model and download it

In [ ]:
new_bal = bal_acc(new_logits[tst] + torch.from_numpy(bias), yva[tst])
out = dict(ckpt) if isinstance(ckpt, dict) else {}
out.update({
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    "classes": CLASSES, "input_size": 227, "autocontrast": True,
    "logit_bias": [float(b) for b in bias],
    "tta_margins": [0.0, 0.1],
    "hf_dataset": HF_DATASET,
    "hf_heldout_balanced_acc": float(new_bal),
    "note": "AlexNet fine-tuned on real photos from Hugging Face (" + HF_DATASET + ") starting from the FER-2013 model.",
})
torch.save(out, "alexnet_emotion_new.pth")
print("saved", os.path.getsize("alexnet_emotion_new.pth") // 1_000_000, "MB")

# Sanity check: load it exactly like the website does.
chk = torch.load("alexnet_emotion_new.pth", map_location="cpu", weights_only=False)
m = build_alexnet(); m.load_state_dict(chk["state_dict"]); m.eval()
with torch.no_grad(): assert m(Xva_in[:4]).shape == (4, 7)
print("OK: loads like the website. Classes:", chk["classes"])

try:
    from google.colab import files
    files.download("alexnet_emotion_new.pth")
except ImportError:
    pass

**Put it on the website**
1. Rename the downloaded `alexnet_emotion_new.pth` to `alexnet_emotion.pth` and copy it over the old one in your project's `models/` folder.
2. In VS Code terminal: `git add models/alexnet_emotion.pth` then `git commit -m "Retrain on Hugging Face photos"` then `git push`.
3. Render rebuilds automatically (a few minutes). Reload the website and test with your camera.